In [93]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
merged_data = pd.read_csv('clean_data/merged_data.csv')
merged_data

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY
0,100026,0,Cash loans,F,N,N,1,450000.0,497520.0,32521.5,...,0.0,0.0,3.0,0.0,-148.0,-1644.0,0.0,0.0,-314.666667,0.0
1,100064,0,Cash loans,F,N,N,0,67500.0,298728.0,15381.0,...,0.0,0.0,2.0,0.0,-339.0,-509.0,0.0,-141.0,-96.500000,4653.0
2,100072,0,Cash loans,M,N,N,0,180000.0,1080000.0,44118.0,...,0.0,0.0,2.0,0.0,-494.0,-630.0,0.0,0.0,-214.500000,0.0
3,100111,0,Cash loans,F,Y,N,1,112500.0,862560.0,27954.0,...,0.0,0.0,12.0,0.0,-270.0,-2257.0,0.0,0.0,-474.916667,0.0
4,100122,0,Cash loans,F,N,N,1,76500.0,808650.0,26217.0,...,0.0,0.0,3.0,0.0,-281.0,-545.0,0.0,0.0,-32.666667,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20099,456145,0,Cash loans,F,N,N,0,162000.0,900000.0,29034.0,...,0.0,0.0,6.0,0.0,-47.0,-1333.0,1.0,0.0,-475.500000,0.0
20100,456164,0,Cash loans,F,N,N,1,112500.0,334152.0,18256.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20101,456220,0,Cash loans,F,N,N,1,81000.0,1350000.0,39474.0,...,0.0,0.0,5.0,0.0,-406.0,-1594.0,0.0,0.0,-458.800000,0.0
20102,456228,0,Cash loans,F,Y,N,0,540000.0,545040.0,35617.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20104 entries, 0 to 20103
Columns: 156 entries, SK_ID_CURR to TOTAL_ANNUITY
dtypes: float64(99), int64(41), object(16)
memory usage: 23.9+ MB


In [95]:
merged_data.select_dtypes('number').max().map('{:,.0f}'.format)

SK_ID_CURR                      456,251
TARGET                                1
CNT_CHILDREN                          9
AMT_INCOME_TOTAL              3,600,000
AMT_CREDIT                    3,375,000
                                ...    
DAYS_CREDIT_OLDEST                   -5
RECENT_LOAN_COUNT                     5
AVERAGE_DAYS_OVERDUE                  0
AVERAGE_DAYS_CREDIT_UPDATE            0
TOTAL_ANNUITY                 8,227,850
Length: 140, dtype: object

In [96]:
import numpy as np

corr = merged_data.corr(numeric_only=True)

# get upper triangle (avoid duplicates & self-correlation)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# filter for high correlation
high_corr = upper.stack().reset_index()
high_corr.columns = ['var1', 'var2', 'correlation']

# keep only > 0.8
high_corr = high_corr[high_corr['correlation'] > 0.8]

print(high_corr)

                             var1                         var2  correlation
272                  CNT_CHILDREN              CNT_FAM_MEMBERS     0.873956
511                    AMT_CREDIT              AMT_GOODS_PRICE     0.986143
2278         REGION_RATING_CLIENT  REGION_RATING_CLIENT_W_CITY     0.944589
3336               APARTMENTS_AVG                ELEVATORS_AVG     0.836808
3341               APARTMENTS_AVG         LIVINGAPARTMENTS_AVG     0.953958
...                           ...                          ...          ...
6669     OBS_30_CNT_SOCIAL_CIRCLE     OBS_60_CNT_SOCIAL_CIRCLE     0.998018
6726     DEF_30_CNT_SOCIAL_CIRCLE     DEF_60_CNT_SOCIAL_CIRCLE     0.862155
8080  CREDIT_TYPE_Consumer credit                 TOTAL_CLOSED     0.899125
8082  CREDIT_TYPE_Consumer credit   COUNT_LOCAL_CURRENCY_LOANS     0.916921
8277                 TOTAL_CLOSED   COUNT_LOCAL_CURRENCY_LOANS     0.919867

[113 rows x 3 columns]


In [97]:
culled_merged_data = merged_data.copy()
culled_merged_data = culled_merged_data.drop(columns=['CNT_CHILDREN'])
culled_merged_data = culled_merged_data.drop(columns=['AMT_GOODS_PRICE'])
culled_merged_data = culled_merged_data.drop(columns=['REGION_RATING_CLIENT_W_CITY'])
culled_merged_data = culled_merged_data.drop(columns=['ELEVATORS_AVG'])
culled_merged_data = culled_merged_data.drop(columns=['OBS_30_CNT_SOCIAL_CIRCLE'])
culled_merged_data = culled_merged_data.drop(columns=['DEF_30_CNT_SOCIAL_CIRCLE'])

In [98]:
import numpy as np

corr = culled_merged_data.corr(numeric_only=True)

# get upper triangle (avoid duplicates & self-correlation)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# filter for high correlation
high_corr = upper.stack().reset_index()
high_corr.columns = ['var1', 'var2', 'correlation']

# keep only > 0.8
high_corr = high_corr[high_corr['correlation'] > 0.8]

print(high_corr)

                             var1                        var2  correlation
2878               APARTMENTS_AVG        LIVINGAPARTMENTS_AVG     0.953958
2879               APARTMENTS_AVG              LIVINGAREA_AVG     0.919886
2882               APARTMENTS_AVG             APARTMENTS_MODE     0.977334
2887               APARTMENTS_AVG              ELEVATORS_MODE     0.821709
2892               APARTMENTS_AVG       LIVINGAPARTMENTS_MODE     0.937175
...                           ...                         ...          ...
5742        LIVINGAPARTMENTS_MEDI              TOTALAREA_MODE     0.855080
5799              LIVINGAREA_MEDI              TOTALAREA_MODE     0.923839
7322  CREDIT_TYPE_Consumer credit                TOTAL_CLOSED     0.899125
7324  CREDIT_TYPE_Consumer credit  COUNT_LOCAL_CURRENCY_LOANS     0.916921
7519                 TOTAL_CLOSED  COUNT_LOCAL_CURRENCY_LOANS     0.919867

[97 rows x 3 columns]


In [99]:
culled_merged_data = culled_merged_data.drop(columns=['LIVINGAPARTMENTS_AVG'])
culled_merged_data = culled_merged_data.drop(columns=['LIVINGAREA_AVG'])
culled_merged_data = culled_merged_data.drop(columns=['APARTMENTS_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['LIVINGAPARTMENTS_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['TOTALAREA_MODE'])

In [100]:
import numpy as np

corr = culled_merged_data.corr(numeric_only=True)

# get upper triangle (avoid duplicates & self-correlation)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# filter for high correlation
high_corr = upper.stack().reset_index()
high_corr.columns = ['var1', 'var2', 'correlation']

# keep only > 0.8
high_corr = high_corr[high_corr['correlation'] > 0.8]

print(high_corr)

                              var1                          var2  correlation
2754                APARTMENTS_AVG                ELEVATORS_MODE     0.821709
2759                APARTMENTS_AVG               LIVINGAREA_MODE     0.900278
2762                APARTMENTS_AVG               APARTMENTS_MEDI     0.996852
2767                APARTMENTS_AVG                ELEVATORS_MEDI     0.834597
2772                APARTMENTS_AVG         LIVINGAPARTMENTS_MEDI     0.951538
2773                APARTMENTS_AVG               LIVINGAREA_MEDI     0.918606
2840              BASEMENTAREA_AVG             BASEMENTAREA_MODE     0.978768
2853              BASEMENTAREA_AVG             BASEMENTAREA_MEDI     0.997116
2929   YEARS_BEGINEXPLUATATION_AVG  YEARS_BEGINEXPLUATATION_MODE     0.975155
2942   YEARS_BEGINEXPLUATATION_AVG  YEARS_BEGINEXPLUATATION_MEDI     0.997595
3018               YEARS_BUILD_AVG              YEARS_BUILD_MODE     0.988541
3031               YEARS_BUILD_AVG              YEARS_BUILD_MEDI

In [101]:
culled_merged_data = culled_merged_data.drop(columns=['LIVINGAREA_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['APARTMENTS_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['ELEVATORS_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['LIVINGAPARTMENTS_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['LIVINGAREA_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['BASEMENTAREA_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['BASEMENTAREA_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['YEARS_BEGINEXPLUATATION_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['YEARS_BEGINEXPLUATATION_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['COMMONAREA_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['COMMONAREA_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['ENTRANCES_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['ENTRANCES_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['FLOORSMAX_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['FLOORSMAX_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['FLOORSMIN_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['FLOORSMIN_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['LANDAREA_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['LANDAREA_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['NONLIVINGAPARTMENTS_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['NONLIVINGAPARTMENTS_MEDI'])

In [102]:
import numpy as np

corr = culled_merged_data.corr(numeric_only=True)

# get upper triangle (avoid duplicates & self-correlation)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# filter for high correlation
high_corr = upper.stack().reset_index()
high_corr.columns = ['var1', 'var2', 'correlation']

# keep only > 0.8
high_corr = high_corr[high_corr['correlation'] > 0.8]

print(high_corr)

                             var1                        var2  correlation
2205               APARTMENTS_AVG              ELEVATORS_MODE     0.821709
2407              YEARS_BUILD_AVG            YEARS_BUILD_MODE     0.988541
2410              YEARS_BUILD_AVG            YEARS_BUILD_MEDI     0.998179
2846            NONLIVINGAREA_AVG          NONLIVINGAREA_MODE     0.943275
2848            NONLIVINGAREA_AVG          NONLIVINGAREA_MEDI     0.991324
2906             YEARS_BUILD_MODE            YEARS_BUILD_MEDI     0.988044
3021           NONLIVINGAREA_MODE          NONLIVINGAREA_MEDI     0.953247
4486  CREDIT_TYPE_Consumer credit                TOTAL_CLOSED     0.899125
4488  CREDIT_TYPE_Consumer credit  COUNT_LOCAL_CURRENCY_LOANS     0.916921
4683                 TOTAL_CLOSED  COUNT_LOCAL_CURRENCY_LOANS     0.919867


In [103]:
culled_merged_data = culled_merged_data.drop(columns=['ELEVATORS_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['YEARS_BUILD_MODE'])
# culled_merged_data = culled_merged_data.drop(columns=['YEARS_BUILD_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['NONLIVINGAREA_MODE'])
culled_merged_data = culled_merged_data.drop(columns=['NONLIVINGAREA_MEDI'])
culled_merged_data = culled_merged_data.drop(columns=['YEARS_BUILD_MEDI'])

In [104]:
import numpy as np

corr = culled_merged_data.corr(numeric_only=True)

# get upper triangle (avoid duplicates & self-correlation)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# filter for high correlation
high_corr = upper.stack().reset_index()
high_corr.columns = ['var1', 'var2', 'correlation']

# keep only > 0.8
high_corr = high_corr[high_corr['correlation'] > 0.8]

print(high_corr)

                             var1                        var2  correlation
4018  CREDIT_TYPE_Consumer credit                TOTAL_CLOSED     0.899125
4020  CREDIT_TYPE_Consumer credit  COUNT_LOCAL_CURRENCY_LOANS     0.916921
4215                 TOTAL_CLOSED  COUNT_LOCAL_CURRENCY_LOANS     0.919867


In [105]:
culled_merged_data

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,NAME_TYPE_SUITE,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY
0,100026,0,Cash loans,F,N,N,450000.0,497520.0,32521.5,Unaccompanied,...,0.0,0.0,3.0,0.0,-148.0,-1644.0,0.0,0.0,-314.666667,0.0
1,100064,0,Cash loans,F,N,N,67500.0,298728.0,15381.0,Family,...,0.0,0.0,2.0,0.0,-339.0,-509.0,0.0,-141.0,-96.500000,4653.0
2,100072,0,Cash loans,M,N,N,180000.0,1080000.0,44118.0,Unaccompanied,...,0.0,0.0,2.0,0.0,-494.0,-630.0,0.0,0.0,-214.500000,0.0
3,100111,0,Cash loans,F,Y,N,112500.0,862560.0,27954.0,Unaccompanied,...,0.0,0.0,12.0,0.0,-270.0,-2257.0,0.0,0.0,-474.916667,0.0
4,100122,0,Cash loans,F,N,N,76500.0,808650.0,26217.0,Unaccompanied,...,0.0,0.0,3.0,0.0,-281.0,-545.0,0.0,0.0,-32.666667,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20099,456145,0,Cash loans,F,N,N,162000.0,900000.0,29034.0,Unaccompanied,...,0.0,0.0,6.0,0.0,-47.0,-1333.0,1.0,0.0,-475.500000,0.0
20100,456164,0,Cash loans,F,N,N,112500.0,334152.0,18256.5,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20101,456220,0,Cash loans,F,N,N,81000.0,1350000.0,39474.0,Unaccompanied,...,0.0,0.0,5.0,0.0,-406.0,-1594.0,0.0,0.0,-458.800000,0.0
20102,456228,0,Cash loans,F,Y,N,540000.0,545040.0,35617.5,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [106]:
culled_merged_data = culled_merged_data.drop(columns=['SK_ID_CURR'])


In [107]:
culled_merged_data = culled_merged_data.drop(columns=['CODE_GENDER'])

In [108]:
cat_cols = culled_merged_data.select_dtypes(include=['object', 'category']).columns.tolist()
print(cat_cols)

['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [109]:
cat_cols = [
    'NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 
    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_TYPE_SUITE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 
    'EMERGENCYSTATE_MODE', 'WEEKDAY_APPR_PROCESS_START'
]

culled_merged_data[cat_cols] = culled_merged_data[cat_cols].fillna('Missing')

culled_merged_data['FLAG_OWN_CAR'] = culled_merged_data['FLAG_OWN_CAR'].map({'Y':1, 'N':0})
culled_merged_data['FLAG_OWN_REALTY'] = culled_merged_data['FLAG_OWN_REALTY'].map({'Y':1, 'N':0})

culled_merged_data = pd.get_dummies(culled_merged_data, columns=cat_cols, drop_first=True)

print(culled_merged_data.dtypes)

TARGET                                    int64
FLAG_OWN_CAR                              int64
FLAG_OWN_REALTY                           int64
AMT_INCOME_TOTAL                        float64
AMT_CREDIT                              float64
                                         ...   
WEEKDAY_APPR_PROCESS_START_SATURDAY        bool
WEEKDAY_APPR_PROCESS_START_SUNDAY          bool
WEEKDAY_APPR_PROCESS_START_THURSDAY        bool
WEEKDAY_APPR_PROCESS_START_TUESDAY         bool
WEEKDAY_APPR_PROCESS_START_WEDNESDAY       bool
Length: 225, dtype: object


In [110]:
culled_merged_data[culled_merged_data.columns] = culled_merged_data[culled_merged_data.columns].fillna(0)

In [111]:
#model with 117 columns 

# sample code

# prepare data for logistic regression
X = culled_merged_data.drop('TARGET', axis = 1)
y = culled_merged_data['TARGET']

# create a logistic regression model and fit the training data
logreg = LogisticRegression(solver='liblinear', class_weight='balanced')
logreg.fit(X, y)

# concatenate intercept and coefficients to a single array
coeff = np.concatenate([logreg.intercept_, logreg.coef_.reshape(-1)])

# create a pandas Series with the features corresponding to the coefficients
coeff_series = pd.Series(coeff, index = ['Intercept'] + X.columns.tolist())
coeff_series

c:\Users\ASUS\anaconda3\envs\is217_env\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Intercept                               9.076063e-03
FLAG_OWN_CAR                           -1.154672e-02
FLAG_OWN_REALTY                         0.000000e+00
AMT_INCOME_TOTAL                       -1.584330e-06
AMT_CREDIT                             -4.358916e-07
                                            ...     
WEEKDAY_APPR_PROCESS_START_SATURDAY     3.334599e-03
WEEKDAY_APPR_PROCESS_START_SUNDAY      -1.893286e-03
WEEKDAY_APPR_PROCESS_START_THURSDAY    -1.055587e-03
WEEKDAY_APPR_PROCESS_START_TUESDAY     -1.555195e-03
WEEKDAY_APPR_PROCESS_START_WEDNESDAY    4.649405e-03
Length: 225, dtype: float64

In [112]:
culled_merged_data

,TARGET,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_Yes,WEEKDAY_APPR_PROCESS_START_MONDAY,WEEKDAY_APPR_PROCESS_START_SATURDAY,WEEKDAY_APPR_PROCESS_START_SUNDAY,WEEKDAY_APPR_PROCESS_START_THURSDAY,WEEKDAY_APPR_PROCESS_START_TUESDAY,WEEKDAY_APPR_PROCESS_START_WEDNESDAY
0,0,0,0,450000.0,497520.0,32521.5,0.020713,-11146,-4306,-114.0,...,False,False,True,False,False,False,False,True,False,False
1,0,0,0,67500.0,298728.0,15381.0,0.019101,-21621,365243,-2019.0,...,False,False,False,False,True,False,False,False,False,False
2,0,0,0,180000.0,1080000.0,44118.0,0.010006,-7907,-1324,-4557.0,...,False,False,True,False,False,False,False,False,True,False
3,0,1,0,112500.0,862560.0,27954.0,0.022800,-10485,-1249,-4628.0,...,True,False,True,False,True,False,False,False,False,False
4,0,0,0,76500.0,808650.0,26217.0,0.022800,-10954,-2469,-550.0,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20099,0,0,0,162000.0,900000.0,29034.0,0.026392,-10592,-2189,-1412.0,...,True,False,True,False,True,False,False,False,False,False
20100,0,0,0,112500.0,334152.0,18256.5,0.022800,-8146,-1197,-2827.0,...,False,False,False,False,False,False,False,False,False,True
20101,0,0,0,81000.0,1350000.0,39474.0,0.024610,-10567,-3855,-1260.0,...,False,True,True,False,False,False,False,False,False,False
20102,0,1,0,540000.0,545040.0,35617.5,0.032561,-12847,-328,-2531.0,...,False,False,False,False,True,False,False,False,False,False


In [114]:
# predict class labels
y_pred = logreg.predict(X)

# convert actual y values to array for comparison with predicted values
y_array = y.to_numpy() 

# compute accuracy
# first, calcuate how many predicted values match the actual values
correct_predictions = sum(y_pred[i] == y_array[i] for i in range(len(y)))

# then, calculate the accurate predictions as a percentage of the total
accuracy = correct_predictions / len(y)
print(f'Model accuracy: {accuracy:.4}')

Model accuracy: 0.5902


In [116]:
culled_merged_data.shape

(20104, 225)

In [117]:
culled_merged_data

,TARGET,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_Yes,WEEKDAY_APPR_PROCESS_START_MONDAY,WEEKDAY_APPR_PROCESS_START_SATURDAY,WEEKDAY_APPR_PROCESS_START_SUNDAY,WEEKDAY_APPR_PROCESS_START_THURSDAY,WEEKDAY_APPR_PROCESS_START_TUESDAY,WEEKDAY_APPR_PROCESS_START_WEDNESDAY
0,0,0,0,450000.0,497520.0,32521.5,0.020713,-11146,-4306,-114.0,...,False,False,True,False,False,False,False,True,False,False
1,0,0,0,67500.0,298728.0,15381.0,0.019101,-21621,365243,-2019.0,...,False,False,False,False,True,False,False,False,False,False
2,0,0,0,180000.0,1080000.0,44118.0,0.010006,-7907,-1324,-4557.0,...,False,False,True,False,False,False,False,False,True,False
3,0,1,0,112500.0,862560.0,27954.0,0.022800,-10485,-1249,-4628.0,...,True,False,True,False,True,False,False,False,False,False
4,0,0,0,76500.0,808650.0,26217.0,0.022800,-10954,-2469,-550.0,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20099,0,0,0,162000.0,900000.0,29034.0,0.026392,-10592,-2189,-1412.0,...,True,False,True,False,True,False,False,False,False,False
20100,0,0,0,112500.0,334152.0,18256.5,0.022800,-8146,-1197,-2827.0,...,False,False,False,False,False,False,False,False,False,True
20101,0,0,0,81000.0,1350000.0,39474.0,0.024610,-10567,-3855,-1260.0,...,False,True,True,False,False,False,False,False,False,False
20102,0,1,0,540000.0,545040.0,35617.5,0.032561,-12847,-328,-2531.0,...,False,False,False,False,True,False,False,False,False,False


In [121]:
# sample code

def woe_iv(data):

    working_data = data.copy() 

    # missing values have been assigned NaN when binning with pd.qcut
    # rename this bin as 'Missing' to consider into WOE calculation
    working_data['Bin_Range'] = working_data['Bin_Range'].astype('object')
    working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')
    
    variable_data = pd.DataFrame()
    variable_data['Bin_Range'] = working_data.groupby(by='Bin_Range', as_index=False).count()['Bin_Range']

    variable_data['Count'] = working_data.groupby(by='Bin_Range', as_index=False).count()['BAD']

    variable_data['Events'] =working_data.groupby(by='Bin_Range', as_index=False).sum()['BAD']

    variable_data['Non_Events'] = variable_data['Count'] - variable_data['Events']

    variable_data['%_of_Events'] = variable_data['Events']/sum(variable_data['Events'])

    variable_data['%_of_Non_Events'] = variable_data['Non_Events']/sum(variable_data['Non_Events'])

    variable_data['WOE'] = np.log(variable_data['%_of_Non_Events'] / variable_data['%_of_Events'])

    variable_data['IV'] = (variable_data['%_of_Non_Events']-variable_data['%_of_Events']) * variable_data['WOE']

    variable_data['total_IV'] = variable_data['IV'].sum()

    return(variable_data)

numeric_cols = culled_merged_data.select_dtypes(include=['float64', 'int64']).drop('TARGET', axis=1).columns

iv_summary = pd.DataFrame(columns=['Variable', 'IV'])

for col in numeric_cols:
    temp = culled_merged_data.loc[:, [col, 'TARGET']].copy()
    temp.rename(columns={'TARGET':'BAD'}, inplace=True)
    
    # bin numeric variable into 10 quantiles
    temp['Bin_Range'] = pd.qcut(temp[col], q=10, duplicates='drop')
    
    # calculate WOE and IV
    iv = woe_iv(temp)
    
    # append IV summary
    iv_summary = pd.concat([iv_summary, pd.DataFrame({'Variable':[col], 'IV':[iv['total_IV'].iloc[0]]})], ignore_index=True)

# sort by IV
iv_summary = iv_summary.sort_values(by='IV', ascending=False)
print(iv_summary)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_29424\1575529717.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')
C:\Users\ASUS\AppData\Local\Temp\ipykernel_29424\1575529717.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  iv_summary = pd.concat([iv_summary, pd.DataFrame({'Variable':[col], 'IV':[iv['total_IV'].iloc[0]]})], ignore_index=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_29424\1575529717.py:10: FutureWarning: Downcasting object dtype arra

                                  Variable        IV
26                            EXT_SOURCE_1  0.094649
7                            DAYS_EMPLOYED  0.066577
97                      DAYS_CREDIT_RECENT  0.065333
3                               AMT_CREDIT  0.064448
101             AVERAGE_DAYS_CREDIT_UPDATE  0.057007
..                                     ...       ...
74   CREDIT_TYPE_Cash loan (non-earmarked)  0.000000
72        CREDIT_TYPE_Another type of loan  0.000000
93                              TOTAL_SOLD  0.000000
94                          TOTAL_BAD_DEBT  0.000000
96            COUNT_FOREIGN_CURRENCY_LOANS  0.000000

[103 rows x 2 columns]


C:\Users\ASUS\AppData\Local\Temp\ipykernel_29424\1575529717.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')


In [122]:
# get list of variables with IV < 0.02
low_iv_cols = iv_summary[iv_summary['IV'] < 0.02]['Variable'].tolist()
print("Columns with IV < 0.02:", low_iv_cols)

Columns with IV < 0.02: ['LANDAREA_AVG', 'TOTAL_CREDIT_LIMIT', 'TOTAL_ACTIVE', 'RECENT_LOAN_COUNT', 'CREDIT_TYPE_Consumer credit', 'COMMONAREA_AVG', 'HOUR_APPR_PROCESS_START', 'NONLIVINGAREA_AVG', 'CREDIT_TYPE_Credit card', 'MAX_OVERDUE_AMOUNT', 'AVERAGE_DAYS_OVERDUE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'COUNT_OVERDUE_LOANS', 'NONLIVINGAPARTMENTS_AVG', 'TOTAL_ANNUITY', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'AMT_REQ_CREDIT_BUREAU_MON', 'CNT_FAM_MEMBERS', 'OWN_CAR_AGE', 'AMT_REQ_CREDIT_BUREAU_QRT', 'REG_REGION_NOT_LIVE_REGION', 'FLAG_CONT_MOBILE', 'FLAG_WORK_PHONE', 'FLAG_EMP_PHONE', 'LIVE_REGION_NOT_WORK_REGION', 'FLAG_OWN_REALTY', 'FLAG_OWN_CAR', 'REG_REGION_NOT_WORK_REGION', 'FLAG_MOBIL', 'REG_CITY_NOT_LIVE_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'REG_CITY_NOT_WORK_CITY', 'FLAG_EMAIL', 'FLAG_PHONE', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_2', 'DEF_60_CNT_SOCIAL_CIRCLE', 'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DO

In [123]:
X = culled_merged_data.drop(columns=['TARGET'] + low_iv_cols)
y = culled_merged_data['TARGET']

In [124]:
# create a logistic regression model and fit the training data
logreg = LogisticRegression(solver='liblinear', class_weight='balanced')
logreg.fit(X, y)

# concatenate intercept and coefficients to a single array
coeff = np.concatenate([logreg.intercept_, logreg.coef_.reshape(-1)])

# create a pandas Series with the features corresponding to the coefficients
coeff_series = pd.Series(coeff, index = ['Intercept'] + X.columns.tolist())
coeff_series

Intercept                              -1.353819e-02
AMT_INCOME_TOTAL                       -1.049211e-06
AMT_CREDIT                             -3.766904e-07
AMT_ANNUITY                             1.284864e-05
REGION_POPULATION_RELATIVE             -4.511232e-03
                                            ...     
WEEKDAY_APPR_PROCESS_START_SATURDAY     1.630330e-02
WEEKDAY_APPR_PROCESS_START_SUNDAY      -3.902686e-02
WEEKDAY_APPR_PROCESS_START_THURSDAY    -2.861025e-02
WEEKDAY_APPR_PROCESS_START_TUESDAY     -3.666907e-02
WEEKDAY_APPR_PROCESS_START_WEDNESDAY    1.803854e-02
Length: 148, dtype: float64

In [125]:
# predict class labels
y_pred = logreg.predict(X)

# convert actual y values to array for comparison with predicted values
y_array = y.to_numpy() 

# compute accuracy
# first, calcuate how many predicted values match the actual values
correct_predictions = sum(y_pred[i] == y_array[i] for i in range(len(y)))

# then, calculate the accurate predictions as a percentage of the total
accuracy = correct_predictions / len(y)
print(f'Model accuracy: {accuracy:.4}')

Model accuracy: 0.6018
